# Workshop 9. Logit and probit for a yes/no outcome

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture09/workshop09.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

Task 1 is the lecture's walkthrough in code, fully worked: run it, answer the questions and change one thing. Tasks 2
to 4 start from an empty cell: predicted churn across visits for app and web users, a likelihood ratio test for
region, and the LPM, logit and probit side by side with your choice between them. The extension tasks are for those
who have time left. Exercises in the style of the textbook, with small numbers and no data, are in the TA's separate
notebook for this session (link on Moodle).

Each core task has two hints: the first names the statistical step, the second names the functions. Try the task
before you open them. The toolbox says where you have seen each function before, so you can look it up in that
notebook. The code patterns, the slides and the lecture notebook are your main references. You are welcome to use an
LLM, but every task can be solved without one, and you should be able to explain the statistics in your answers
yourself.

In every task, say what y and the x's are before you fit anything. Read the sign and significance from the
coefficients, and the sizes as probabilities or percentage points. Each answer is in words a manager can follow, with
the numbers it rests on.

## Toolbox

The functions you need today, and where you have seen them in use.

| Function | What it does | Seen in |
|---|---|---|
| `smf.logit("y ~ x1 + x2", data=df).fit()` | Fits a logit model by maximum likelihood; y is 0 or 1 | Lecture 9, Part 2 |
| `smf.probit(...)` | The same with the normal curve | Lecture 9, Part 4 |
| `smf.ols(...).fit(cov_type="HC1")` | The linear probability model with robust standard errors | Lecture 8, Part 4 |
| `model.summary()`, `model.params`, `model.pvalues` | The output, the coefficients, the p-values | Lecture 6; Lecture 9, Part 2 |
| `model.llf`, `model.prsquared` | The log-likelihood and McFadden's pseudo R² | Lecture 9, Part 2 |
| `stats.chi2.sf(LR, q)` | The p-value of a likelihood ratio test with q degrees of freedom | Lecture 9, Part 2 (Exercise 2) |
| `model.predict(pd.DataFrame({...}))` | Predicted probabilities; one column for every x in the formula | Lecture 8, Part 4; Lecture 9, Part 3 |
| `model.get_margeff(at="overall", dummy=True)` | Average marginal effects; `.summary()` or `.summary_frame()` | Lecture 9, Part 4 |
| `model.fittedvalues` | ŷ, one per row; in an LPM the predicted probabilities | Lecture 8, Part 4 |
| `np.arange(0, 91, 5)` | The numbers 0, 5, ..., 90 | Lecture 9, Part 3 (the gap figure) |
| `df[df["x"].isin([10, 30, 60])]`, `df.sort_values("x", ascending=False).head(200)` | Rows with given values of x; the 200 rows with the largest x | New today: see the first code pattern |

## Code patterns

Two complete patterns for today. Replace `df`, `y01`, `d1`, `g`, `x` and the values with your data and column names.
The patterns are templates; they do not run as they stand.

**A logit model, predictions for described cases and across a grid, and average marginal effects**

```python
logit = smf.logit("y01 ~ d1 + C(g) + x", data=df).fit()
print(logit.summary())                     # signs, z, p; converged; log-likelihood; pseudo R2; LLR p-value

new = pd.DataFrame({"d1": [0, 1], "g": "A", "x": 3})       # the same case with d1 = 0 and d1 = 1
p = logit.predict(new)
print(p.round(3))
print(f"Gap: {100 * (p.iloc[0] - p.iloc[1]):.1f} percentage points")      # d1 = 0 minus d1 = 1

grid = pd.DataFrame({"d1": 0, "g": "A", "x": np.arange(0, 91, 5)})   # one row per value of x
grid["p"] = logit.predict(grid)
plt.plot(grid["x"], grid["p"])
print(grid[grid["x"].isin([10, 30, 60])])                          # only the rows for these values of x

df["p"] = logit.predict(df)
top = df.sort_values("p", ascending=False).head(200)              # the 200 rows with the highest p

ame = logit.get_margeff(at="overall", dummy=True)
print(ame.summary())                       # dy/dx times 100: percentage points
```

**A likelihood ratio test for a group of x's, and three models side by side**

```python
small = smf.logit("y01 ~ d1 + x", data=df).fit(disp=0)
large = smf.logit("y01 ~ d1 + x + C(g)", data=df).fit(disp=0)          # the same rows, more x's
LR = 2 * (large.llf - small.llf)
q = len(large.params) - len(small.params)                              # number of x's added
print(f"LR = {LR:.2f}, df = {q}, p = {stats.chi2.sf(LR, q):.3f}")

formula = "y01 ~ d1 + C(g) + x"
lpm = smf.ols(formula, data=df).fit(cov_type="HC1")
logit = smf.logit(formula, data=df).fit(disp=0)
probit = smf.probit(formula, data=df).fit(disp=0)
table = pd.DataFrame({
    "LPM": lpm.params.drop("Intercept"),
    "logit AME": logit.get_margeff(at="overall", dummy=True).margeff,
    "probit AME": probit.get_margeff(at="overall", dummy=True).margeff})
print((100 * table).round(2))                                          # percentage points
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 RetailCorp customers drawn at random
customers.head()

## Core tasks

### Task 1. Customer A and the average gap for members (worked)

The lecture's walkthrough in code. Run the cells and answer the questions below them.

y = `churned` (1 if no order in January to June 2026); the x's are `member`, `channel`, `tenure_years`, `age` and
`visits_2025`, as in the Lecture 8 LPM.

In [ ]:
m_logit = smf.logit("churned ~ member + C(channel) + tenure_years + age + visits_2025", data=customers).fit()
print(m_logit.summary())

In [ ]:
channel = "App"      # change to "Web"

customer_a = pd.DataFrame({"member": [0, 1], "channel": channel, "tenure_years": 1, "age": 45, "visits_2025": 15})
p = m_logit.predict(customer_a)
print(f"Not a member: {p.iloc[0]:.3f}")
print(f"Member:       {p.iloc[1]:.3f}")
print(f"Gap: {100 * (p.iloc[0] - p.iloc[1]):.1f} percentage points")

In [ ]:
ame = m_logit.get_margeff(at="overall", dummy=True)
print(ame.summary())

**Questions**

1. What do the signs and p-values of the coefficients tell you? Why do we not read 0.4932 on Web as a change in
   probability?
2. What are the log-likelihood, LL-Null and the pseudo R², and what does the LLR p-value test?
3. Read the predicted probabilities for customer A, and the average marginal effect of membership with its interval.
   Why is the gap for customer A larger than the average marginal effect?

**Change one thing.** In the second cell, change `channel` to `"Web"` and run it. How do the two probabilities and
the gap change, and why?

**Your answer:**

### Task 2. Predicted churn across visits for app and web users

The marketing team wants to see how the probability of churn falls with visits, for app and for web users. Use
`m_logit` from Task 1.

1. Make a table of predicted probabilities for non-members with 3 years as customers, aged 45, for visits from 0 to
   90 in steps of 5, once for app users and once for web users. Plot the two curves in one figure.
2. Read the difference between web and app users, in percentage points, at 10, 30 and 60 visits.
3. Why does the difference change with visits, when the model has one coefficient for Web?

<details>
<summary>Hint 1</summary>

A predicted probability is the model's answer for a customer you describe. Describe the same customer at many values
of visits, once with channel App and once with channel Web, and the curves show how the probability changes. In a
logit model the same change in z moves the probability most near 0.5.
</details>

<details>
<summary>Hint 2</summary>

The first code pattern: `pd.DataFrame` with `"visits_2025": np.arange(0, 91, 5)` and the other columns fixed, then
`m_logit.predict(grid)`. Do it once with `"channel": "App"` and once with `"channel": "Web"`. Print the rows for 10,
30 and 60 visits with `.isin([10, 30, 60])`, as in the first code pattern.
</details>

**Your answer:**

### Task 3. Does region help to predict churn?

1. Add `C(region)` to the model of Task 1. How many coefficients does that add, and which region is the reference?
2. Use a likelihood ratio test to answer whether the region dummies together help to predict churn, given the other
   x's. Give LR, the degrees of freedom and the p-value.
3. Compute the average marginal effects of the larger model. What do they say about the regions, and does membership
   change?

<details>
<summary>Hint 1</summary>

The likelihood ratio test compares the log-likelihoods of two models fitted on the same customers: the larger model
with region and the smaller one without. LR = 2 (ℓ larger − ℓ smaller) has a χ² distribution with as many degrees of
freedom as the number of coefficients added.
</details>

<details>
<summary>Hint 2</summary>

The second code pattern: `m_region = smf.logit(... + C(region) ..., data=customers).fit()`, then
`2 * (m_region.llf - m_logit.llf)` and `stats.chi2.sf(LR, 4)`. For the AMEs, `get_margeff(at="overall", dummy=True)`.
</details>

**Your answer:**

### Task 4. The LPM, logit and probit side by side

Fit the model of Task 1 as an LPM (with robust standard errors), a logit and a probit model.

1. How many predicted probabilities are below 0 or above 1 in each model?
2. Put the LPM coefficients and the logit and probit average marginal effects side by side, in percentage points. How
   close are they?
3. Predict the probability of churn in all three models for two customers: (C) a web customer who is not a member,
   aged 30, with half a year as a customer and 8 visits; (D) an app user who is a member, aged 60, with 9 years and 70
   visits.
4. In three or four sentences: which model would you report to RetailCorp's management, and why?

<details>
<summary>Hint 1</summary>

The three models answer the same question with different curves: a straight line, the logistic curve and the normal
curve. Compare them where it matters: the averages, and the predictions for customers far from the middle of the
data.
</details>

<details>
<summary>Hint 2</summary>

The second code pattern for the three models and the table. `model.fittedvalues` for the LPM and
`model.predict(customers)` for logit and probit give one predicted probability per customer; count the values below 0
with `(p < 0).sum()`. For the two customers, one `pd.DataFrame` with two rows and `model.predict(new)` in each model.
</details>

**Your answer:**

## Extension tasks

### Extension 1. A retention offer for 200 customers

RetailCorp can make a retention offer to 200 customers. Take the 200 customers with the highest predicted probability
of churn in `m_logit` and count how many of them churned. How many churners would 200 customers drawn at random
contain on average? What does the comparison say about the model's use for this decision?

**Your answer:**

### Extension 2. Odds ratios, and why we translate them

`np.exp(m_logit.params["member"])` is the odds ratio for membership. Compute it, and compute the odds
p / (1 − p) for customer A (Task 1, app user) and for customer B (app user, aged 45, 8 years, 60 visits), as a
non-member and as a member. What is the same for A and B, and what is not?

**Your answer:**